# 04 · The vitals model as the app will use it

`VitalsModel.assess()` takes one patient's vitals and returns a JSON-ready result:
NEWS2 with the safety extension, `p_urgent` from the classifier, a safety floor, and a one-line note for the main
language model. This notebook shows it on example patients and how the floor is enforced on the main model's answer.

Interface details for the app: see `../README.md`.

In [1]:
import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
from vitals_model import VitalsModel, enforce_floor

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 220)
vm = VitalsModel()
print("threshold:", vm.threshold, "| floor if flagged:", vm.floor_if_flagged)

threshold: 0.1401 | floor if flagged: 3


## 1. Example patients

A–C are the examples from the group plan; D–F test the hard cases.

In [2]:
patients = {
    "A: breathless, low oxygen": dict(rr=26, spo2=89, sbp=112, hr=124, temp_c=38.6, consciousness="alert",
                                      age=71, sex="male", pain_score=2),
    "B: fall, painful leg":      dict(rr=18, spo2=98, sbp=138, hr=92, temp_c=36.8, consciousness="alert",
                                      age=45, sex="female", pain_score=7, injury=True),
    "C: minor complaint":        dict(rr=16, spo2=99, sbp=124, hr=74, temp_c=36.6, consciousness="alert",
                                      age=28, sex="male", pain_score=2),
    "D: found unresponsive":     dict(consciousness="unresponsive", age=63, sex="male"),
    "E: temperature in °F":      dict(rr=20, spo2=96, sbp=104, hr=108, temp_c=102.2, consciousness="alert", age=34),
    "F: SpO2 typo (9%)":         dict(rr=18, spo2=9, sbp=130, hr=80, temp_c=37.0, consciousness="alert", age=50),
}
results = {name: vm.assess(**p) for name, p in patients.items()}
pd.DataFrame({name: {k: r[k] for k in ["news2", "news2_risk", "p_urgent", "urgent_flag", "floor_priority",
                                        "danger", "missing", "warnings"]}
              for name, r in results.items()}).T

,news2,news2_risk,p_urgent,urgent_flag,floor_priority,danger,missing,warnings
"A: breathless, low oxygen",9,high,0.819,True,2,True,[],[]
"B: fall, painful leg",1,low,0.171,True,3,False,[],[]
C: minor complaint,0,low,0.104,False,5,False,[],[]
D: found unresponsive,3,high,0.947,True,2,True,"[rr, spo2, sbp, hr, temp_c]",[]
E: temperature in °F,3,low,0.517,True,3,False,[],[temperature 102.2 read as °F -> 39.0 °C]
F: SpO2 typo (9%),0,low,0.114,False,5,False,[spo2],[spo2 = 9 is outside 50-100; treated as not recorded]


In [3]:
for name, r in results.items():
    print(f"{name}\n  {r['note']}\n")

A: breathless, low oxygen
  NEWS2 9 = HIGH risk. Abnormal: RR 26/min (+3), SpO2 89% (+3), HR 124 bpm (+2), Temp 38.6 °C (+1). Vitals model: 82% chance of needing KTAS 1-2 care (above the 14% alert threshold). Safety floor: priority 2 or more urgent.

B: fall, painful leg
  NEWS2 1 = LOW risk. Abnormal: HR 92 bpm (+1). Vitals model: 17% chance of needing KTAS 1-2 care (above the 14% alert threshold). Safety floor: priority 3 or more urgent.

C: minor complaint
  NEWS2 0 = LOW risk. All recorded vitals in normal range. Vitals model: 10% chance of needing KTAS 1-2 care.

D: found unresponsive
  NEWS2 3 = HIGH risk (escalated: core vitals could not be recorded; unresponsive). Abnormal: unresponsive. Not recorded: RR, SpO2, SBP, HR, Temp. Vitals model: 95% chance of needing KTAS 1-2 care (above the 14% alert threshold). Safety floor: priority 2 or more urgent.

E: temperature in °F
  NEWS2 3 = LOW risk. Abnormal: SBP 104 mmHg (+1), HR 108 bpm (+1), Temp 39 °C (+1). Vitals model: 52% chance 

Full output for one patient (this is what the app stores and passes on):

In [4]:
print(json.dumps(results["A: breathless, low oxygen"], indent=2))

{
  "news2": 9,
  "news2_risk": "high",
  "subscores": {
    "rr": 3,
    "spo2": 3,
    "sbp": 0,
    "hr": 2,
    "temp_c": 1,
    "consciousness": 0,
    "on_oxygen": 0
  },
  "escalated": false,
  "escalation_reason": "",
  "p_urgent": 0.819,
  "urgent_flag": true,
  "floor_priority": 2,
  "danger": true,
  "missing": [],
  "assumed": [
    "on_oxygen"
  ],
  "warnings": [],
  "note": "NEWS2 9 = HIGH risk. Abnormal: RR 26/min (+3), SpO2 89% (+3), HR 124 bpm (+2), Temp 38.6 \u00b0C (+1). Vitals model: 82% chance of needing KTAS 1-2 care (above the 14% alert threshold). Safety floor: priority 2 or more urgent."
}


## 2. Enforcing the safety floor on the main model's answer

The main language model proposes a priority; `enforce_floor` makes sure it is never less urgent than the floor.
Below, a pretend LLM output that ranked the unresponsive patient as priority 5 gets corrected.

In [5]:
llm_priority = {"A: breathless, low oxygen": 1, "B: fall, painful leg": 3, "C: minor complaint": 5,
                "D: found unresponsive": 5, "E: temperature in °F": 3, "F: SpO2 typo (9%)": 4}
board = pd.DataFrame({
    "LLM priority": llm_priority,
    "vitals floor": {n: r["floor_priority"] for n, r in results.items()},
    "p_urgent": {n: r["p_urgent"] for n, r in results.items()},
})
board["final priority"] = [enforce_floor(l, f) for l, f in zip(board["LLM priority"], board["vitals floor"])]
board["changed by floor"] = board["final priority"] != board["LLM priority"]
board.sort_values(["final priority", "p_urgent"], ascending=[True, False])

,LLM priority,vitals floor,p_urgent,final priority,changed by floor
"A: breathless, low oxygen",1,2,0.819,1,False
D: found unresponsive,5,2,0.947,2,True
E: temperature in °F,3,3,0.517,3,False
"B: fall, painful leg",3,3,0.171,3,False
F: SpO2 typo (9%),4,5,0.114,4,False
C: minor complaint,5,5,0.104,5,False


Ranked list for the doctor: sort by final priority, then by `p_urgent` within the same priority.

## 3. Speed

In [6]:
t = time.perf_counter()
for _ in range(200):
    vm.assess(**patients["A: breathless, low oxygen"])
print(f"{(time.perf_counter() - t) / 200 * 1000:.1f} ms per patient")

5.7 ms per patient


## 4. Group decision: classifier floor on or off?

`VitalsModel(use_classifier_floor=True)` (default) lets the classifier tighten the floor to priority 3 when
`p_urgent` ≥ threshold. On the test set (notebook 03) this doubled the urgent patients protected by a floor
(15 → 30 of 37) but raised over-triaged visits from 3 to 28 of 190. `use_classifier_floor=False` keeps only the
NEWS2-based floor. The classifier's `p_urgent` and note are passed to the main model either way.

In [7]:
strict = VitalsModel(use_classifier_floor=False)
pd.DataFrame({"floor (classifier on)": {n: r["floor_priority"] for n, r in results.items()},
              "floor (classifier off)": {n: strict.assess(**p)["floor_priority"] for n, p in patients.items()}})

,floor (classifier on),floor (classifier off)
"A: breathless, low oxygen",2,2
"B: fall, painful leg",3,5
C: minor complaint,5,5
D: found unresponsive,2,2
E: temperature in °F,3,5
F: SpO2 typo (9%),5,5
